# 第5章 回归模型 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]（仅误差与泛化差距口径）
- [[课程/深度学习/演示代码/Chapter5.py]]
- [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q5A 30分钟综合题：五维响应函数回归

**新编模拟，依据** [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]“学习d维目标函数”小节、[[课程/深度学习/演示代码/Chapter5.py]]。情境：用离线采样点拟合平滑响应场，不使用外部数据。

**共100分，30分钟，从空白独立编程。**

1. **数据（15分）**：导入所需库，CPU单线程、torch.manual_seed(51)。实现`target_a5(x)`，输入(N,5)，返回(N,)，值为1/(1+sum(x²))。顺序采样512条训练、128条测试，均匀于[-1,1]^5；命名Xa_train、ya_train、Xa_test、ya_test，测试不能复制训练。
2. **网络（20分）**：实现`build_reg_a()`，5→16→16→1、隐藏ReLU、输出无激活；命名net_a5。打印参数量，N=1输出必须(1,1)。
3. **训练（30分）**：Adam，90步全批MSE；学习率含端点地从.01线性降到.001。每步清梯度、重算前向、backward、step，保存更新前loss为history_a5，实际学习率为rates_a5。
4. **评估（20分）**：no_grad重算最终训练MSE、训练/测试RMSE及课程gap_abs=abs(test_rmse−train_rmse)，存字典report_a5，键train_mse/train_rmse/test_rmse/gap_abs。不得混用MSE与RMSE。
5. **图与说明（15分）**：创建output目录，保存output/Q5A.png，左图训练MSE-步数，右图测试真值-预测散点并加y=x参考线，英文轴名。输出一句话：小gap不等于低测试误差；解释为何输出需squeeze(-1)而非直接与(N,)标签广播相减。

允许用nn.Sequential或nn.Module。评分按功能分配，不以某个固定训练性能/浮点数给分。


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

torch.manual_seed(51)
def target_a5(x):
    return 1/(1+x.square().sum(dim=1))
def build_reg_a():
    return nn.Sequential(nn.Linear(5,16), nn.ReLU(), nn.Linear(16,16), nn.ReLU(), nn.Linear(16,1))
Xa_train = 2*torch.rand(512,5)-1
Xa_test = 2*torch.rand(128,5)-1
ya_train, ya_test = target_a5(Xa_train), target_a5(Xa_test)
net_a5 = build_reg_a()
print("Parameter count:", sum(p.numel() for p in net_a5.parameters()))
opt_a5 = torch.optim.Adam(net_a5.parameters(), lr=.01)
rates_a5, history_a5 = np.linspace(.01,.001,90), []
for rate in rates_a5:
    for group in opt_a5.param_groups:
        group["lr"] = float(rate)
    opt_a5.zero_grad()
    loss = (net_a5(Xa_train).squeeze(-1)-ya_train).square().mean()
    loss.backward()
    opt_a5.step()
    history_a5.append(loss.item())
net_a5.eval()
with torch.no_grad():
    train_pred_a5 = net_a5(Xa_train).squeeze(-1)
    test_pred_a5 = net_a5(Xa_test).squeeze(-1)
    train_mse_a5 = (train_pred_a5-ya_train).square().mean().item()
    train_rmse_a5 = np.sqrt(train_mse_a5)
    test_rmse_a5 = (test_pred_a5-ya_test).square().mean().sqrt().item()
report_a5 = dict(train_mse=train_mse_a5, train_rmse=float(train_rmse_a5),
                 test_rmse=test_rmse_a5, gap_abs=abs(test_rmse_a5-train_rmse_a5))
Path("output").mkdir(exist_ok=True)
fig, axes = plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(history_a5)
axes[0].set(xlabel="Update", ylabel="Training MSE")
axes[1].scatter(ya_test.numpy(), test_pred_a5.numpy(), s=8)
lo, hi = min(ya_test.min().item(),test_pred_a5.min().item()), max(ya_test.max().item(),test_pred_a5.max().item())
axes[1].plot([lo,hi],[lo,hi],"k--")
axes[1].set(xlabel="True response", ylabel="Predicted response")
fig.tight_layout()
fig.savefig("output/Q5A.png", dpi=120)
plt.close(fig)
print(report_a5)
print("小gap不代表低误差；squeeze(-1)消去输出维以免(N,1)-(N,)广播成(N,N)。")


### 解析与点评

参数量96+272+17=385。顺序为clear→new forward→loss→backward→step。最终评估用更新后模型。图检查拟合趋势与偏差，不仅看下降曲线；测试阈值只是宽松冒烟检查，不是考试的性能标准。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
assert Xa_train.shape == (512,5) and Xa_test.shape == (128,5)
assert torch.allclose(target_a5(torch.zeros(1,5)), torch.ones(1))
assert torch.allclose(target_a5(torch.ones(2,5)), torch.full((2,),1/6))
assert net_a5(torch.zeros(1,5)).shape == (1,1)
assert sum(p.numel() for p in net_a5.parameters()) == 385
assert len(history_a5) == len(rates_a5) == 90 and np.isfinite(history_a5).all()
assert np.allclose([rates_a5[0], rates_a5[-1]], [.01,.001])
assert np.isclose(report_a5["train_rmse"]**2, report_a5["train_mse"], rtol=1e-5)
assert np.isclose(report_a5["gap_abs"], abs(report_a5["test_rmse"]-report_a5["train_rmse"]))
assert np.isfinite(list(report_a5.values())).all() and report_a5["test_rmse"] < 2
assert Path("output/Q5A.png").stat().st_size > 1000

print("自检通过")
